##**Imports**

In [ ]:
# Imports
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
# Set pandas as the default output for sklearn
from sklearn import set_config
set_config(transform_output='pandas')

##**Define Custom Functions**

In [ ]:
def regression_metrics(y_true, y_pred, label='', verbose = True, output_dict=False):
  # Get metrics
  mae = mean_absolute_error(y_true, y_pred)
  mse = mean_squared_error(y_true, y_pred)
  rmse = mean_squared_error(y_true, y_pred)
  r_squared = r2_score(y_true, y_pred)
  if verbose == True:
    # Print Result with Label and Header
    header = "-"*60
    print(header, f"Regression Metrics: {label}", header, sep='\n')
    print(f"- MAE = {mae:,.3f}")
    print(f"- MSE = {mse:,.3f}")
    print(f"- RMSE = {rmse:,.3f}")
    print(f"- R^2 = {r_squared:,.3f}")
  if output_dict == True:
      metrics = {'Label':label, 'MAE':mae,
                 'MSE':mse, 'RMSE':rmse, 'R^2':r_squared}
      return metrics

def evaluate_regression(reg, X_train, y_train, X_test, y_test, verbose = True,
                        output_frame=False):
  # Get predictions for training data
  y_train_pred = reg.predict(X_train)

  # Call the helper function to obtain regression metrics for training data
  results_train = regression_metrics(y_train, y_train_pred, verbose = verbose,
                                     output_dict=output_frame,
                                     label='Training Data')
  print()
  # Get predictions for test data
  y_test_pred = reg.predict(X_test)
  # Call the helper function to obtain regression metrics for test data
  results_test = regression_metrics(y_test, y_test_pred, verbose = verbose,
                                  output_dict=output_frame,
                                    label='Test Data' )

  # Store results in a dataframe if ouput_frame is True
  if output_frame:
    results_df = pd.DataFrame([results_train,results_test])
    # Set the label as the index
    results_df = results_df.set_index('Label')
    # Set index.name to none to get a cleaner looking result
    results_df.index.name=None
    # Return the dataframe
    return results_df.round(3)

##**Load Data**

#**الخطوة 1: ربط المجلدات**

In [ ]:
# Mount google drive
# لربط حسابك بـ Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Load Ames data from AXSOSACADEMY file structure
fpath = "/content/drive/MyDrive/AXSOSACADEMY/01-Fundamentals/Week04/Data/ames-housing-for-ml.csv"
df = pd.read_csv(fpath)
df = df.set_index("PID")
df.head()


,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Utilities,Neighborhood,Bldg Type,House Style,Overall Qual,...,Garage Area,Garage Qual,Garage Cond,Paved Drive,Fence,SalePrice,Month,Year,Total Half Baths,Total Full Baths
PID,,,,,,,,,,,,,,,,,,,,,
907227090,RL,60.0,7200,Pave,NaN,AllPub,CollgCr,1Fam,1Story,5,...,297.0,TA,TA,Y,MnPrv,119900.0,3,2006,0.0,1.0
527108010,RL,134.0,19378,Pave,NaN,AllPub,Gilbert,1Fam,2Story,7,...,576.0,TA,TA,Y,NaN,320000.0,3,2006,1.0,3.0
534275170,RL,NaN,12772,Pave,NaN,AllPub,NAmes,1Fam,1Story,6,...,301.0,TA,TA,Y,NaN,151500.0,4,2007,0.0,1.0
528104050,RL,114.0,14803,Pave,NaN,AllPub,NridgHt,1Fam,1Story,10,...,1220.0,TA,TA,Y,NaN,385000.0,6,2008,0.0,3.0
533206070,FV,32.0,3784,Pave,Pave,AllPub,Somerst,TwnhsE,1Story,8,...,476.0,TA,TA,Y,NaN,193800.0,2,2007,0.0,3.0


**Define X and y and Perform Validation Split**

In [ ]:
# Make a list of features to drop
drop_from_model = ['Utilities', # Quasi-constant
                   "Street", # Quasi-constant
                   'MS Zoning', # Stakeholder can't change
                   'Lot Frontage',  # Stakeholder can't change
                   'Lot Area', # Stakeholder can't change
                   'Neighborhood',  # Stakeholder can't change
                   'Year Built'] # Stakeholder can't change
# Define features matrix
X = df.drop(columns = [*drop_from_model,'SalePrice'])
# Define target
y = df['SalePrice']
# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42)
# Preview training data
X_train.head()

,Alley,Bldg Type,House Style,Overall Qual,Overall Cond,Year Remodeled,Exter Qual,Exter Cond,Bsmt Unf Sqft,Total Bsmnt Sqft,...,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Fence,Month,Year,Total Half Baths,Total Full Baths
PID,,,,,,,,,,,,,,,,,,,,,
905475520,NaN,1Fam,1Story,4,5,1994,TA,TA,0.0,0.0,...,1.0,308.0,TA,TA,N,NaN,8,2007,0.0,1.0
909254010,NaN,1Fam,2Story,7,8,1990,TA,TA,600.0,600.0,...,1.0,215.0,Fa,TA,Y,MnPrv,5,2009,0.0,1.0
531450090,NaN,1Fam,1Story,6,5,1991,TA,TA,78.0,1278.0,...,2.0,496.0,TA,TA,Y,GdWo,6,2008,0.0,3.0
903400040,Pave,1Fam,2Story,6,6,1950,TA,TA,764.0,764.0,...,2.0,520.0,TA,TA,N,GdPrv,7,2007,0.0,1.0
527107130,NaN,1Fam,SLvl,7,5,1997,TA,TA,100.0,384.0,...,2.0,390.0,TA,TA,Y,NaN,6,2009,1.0,2.0


**Create Pipelines for Transformation**

In [ ]:
# PREPROCESSING PIPELINE FOR NUMERIC DATA
# Save list of column names
num_cols = X_train.select_dtypes("number").columns
print("Numeric Columns:", num_cols)
# instantiate preprocessors
impute_median = SimpleImputer(strategy='median')
scaler = StandardScaler()
# Make a numeric preprocessing pipeline
num_pipe = make_pipeline(impute_median, scaler)
# Making a numeric tuple for ColumnTransformer
num_tuple = ('numeric', num_pipe, num_cols)

Numeric Columns: Index(['Overall Qual', 'Overall Cond', 'Year Remodeled', 'Bsmt Unf Sqft',
       'Total Bsmnt Sqft', 'Living Area Sqft', 'Bedroom', 'Kitchen',
       'Total Rooms', 'Garage Yr Blt', 'Garage Cars', 'Garage Area', 'Month',
       'Year', 'Total Half Baths', 'Total Full Baths'],
      dtype='object')


In [ ]:
# Save list of column names
ord_cols = ['Exter Qual','Exter Cond', 'Garage Qual',"Garage Cond"]
print("Ordinal Columns:", ord_cols)
# Create imputer for ordinal data
impute_na_ord = SimpleImputer(strategy='constant', fill_value='NA')
# Making the OrdinalEncoder
# Specifying order of categories for our  Ordinal Qual/Cond Columms
qual_cond_order = ['NA','Po', 'Fa', 'TA', 'Gd', 'Ex']
# Making the list of order lists for OrdinalEncoder
ordinal_category_orders = [qual_cond_order, qual_cond_order,
                           qual_cond_order, qual_cond_order]
ord_encoder = OrdinalEncoder(categories=ordinal_category_orders)
# Making a final scaler to scale category #'s
scaler_ord = StandardScaler()
# Making an ord_pipe
ord_pipe = make_pipeline(impute_na_ord, ord_encoder, scaler_ord)
# Making an ordinal_tuple for ColumnTransformer
ord_tuple = ('ordinal', ord_pipe, ord_cols)

Ordinal Columns: ['Exter Qual', 'Exter Cond', 'Garage Qual', 'Garage Cond']


In [ ]:
# PREPROCESSING PIPELINE FOR ONE-HOT-ENCODED DATA
# Save list of column names
ohe_cols = X_train.select_dtypes('object').drop(columns=ord_cols).columns
print("OneHotEncoder Columns:", ohe_cols)
# Instantiate the individual preprocessors
impute_na = SimpleImputer(strategy='constant', fill_value = "NA")
ohe_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# Make pipeline with imputer and encoder
ohe_pipe = make_pipeline(impute_na, ohe_encoder)
# Making a ohe_tuple for ColumnTransformer
ohe_tuple = ('categorical', ohe_pipe, ohe_cols)

OneHotEncoder Columns: Index(['Alley', 'Bldg Type', 'House Style', 'Central Air', 'Garage Type',
       'Paved Drive', 'Fence'],
      dtype='object')


**Instantiate the Column Transformer**

In [ ]:
# Create the Column Transformer
preprocessor = ColumnTransformer([num_tuple, ord_tuple, ohe_tuple],
                                     verbose_feature_names_out=False)

**Instantiate Model**

In [ ]:
# We will start with a default model
# For reproducible results, set the random state
dec_tree = DecisionTreeRegressor(random_state = 42)

**Create Model Pipeline**

In [ ]:
# Combine the preprocessor with the decision tree model in a model pipeline
dec_tree_pipe = make_pipeline(preprocessor, dec_tree)

In [ ]:
# Fit the model on the training data only
dec_tree_pipe.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  Index(['Overall Qual', 'Overall Cond', 'Year Remodeled', 'Bsmt Unf Sqft',
       'Total Bsmnt Sqft', 'Living Area Sqft', 'Bedroom', 'Kitchen',
       'Total Rooms', 'Garage Yr Blt', 'Garage Cars', 'Garag...
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(fill_value='NA',
                                                                                 strategy='constant')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['Alley', 'Bldg Type', 'House Style', 'Central Air', 'Garage Type',
       'Paved Drive', 'Fence'],
      dtype='object'))],
                                   verbose_feature_names_out=False)),
                ('decisiontreeregressor',
                 DecisionTreeRegressor(random_state=42))])

In [ ]:
# Make predictions and evalute with custom function
evaluate_regression(dec_tree_pipe, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 3.778
- MSE = 15,678.198
- RMSE = 15,678.198
- R^2 = 1.000

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 24,500.764
- MSE = 1,609,120,231.370
- RMSE = 1,609,120,231.370
- R^2 = 0.671



**GridSearchCV with Model Pipelines**

**Select Parameters for Tuning and Define the Param Grid**

In [ ]:
# Looking at options for tuning this model
dec_tree_pipe.get_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[('numeric',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='median')),
                                                    ('standardscaler',
                                                     StandardScaler())]),
                                    Index(['Overall Qual', 'Overall Cond', 'Year Remodeled', 'Bsmt Unf Sqft',
          'Total Bsmnt Sqft', 'Living Area Sqft', 'Bedroom', 'Kitchen',
          'Total Rooms', 'Garage Yr Blt', 'Garage Cars', 'Garage Area', 'Month',
          'Year', 'Total Half Baths...
                                    ['Exter Qual', 'Exter Cond', 'Garage Qual',
                                     'Garage Cond']),
                                   ('categorical',
                                    Pipeline(steps=[('simpleimputer',
                                           

In [ ]:
# Define dictionary of parameters to tune and the values to try
param_grid = {'decisiontreeregressor__max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, None],
              'decisiontreeregressor__min_samples_leaf': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
              'decisiontreeregressor__min_samples_split': [2, 3, 4]}

**Instantiate GridSearchCV**

In [ ]:
# Instantiate GridSearchCV
grid_search = GridSearchCV(dec_tree_pipe, param_grid, n_jobs = -1, verbose = 1)

**Fit the GridSearch on the Training Data**

In [ ]:
# Fit the Gridsearch on the training data
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 330 candidates, totalling 1650 fits


GridSearchCV(estimator=Pipeline(steps=[('columntransformer',
                                        ColumnTransformer(transformers=[('numeric',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('standardscaler',
                                                                                          StandardScaler())]),
                                                                         Index(['Overall Qual', 'Overall Cond', 'Year Remodeled', 'Bsmt Unf Sqft',
       'Total Bsmnt Sqft', 'Living Area Sqft', 'Bedroom', 'Kitchen',
       'Total Rooms', 'Garage Yr B...
       'Paved Drive', 'Fence'],
      dtype='object'))],
                                                          verbose_feature_names_out=False)),
                                       ('decisiontreeregressor',
                                        DecisionTreeRegressor(random_state=42))]),
             n_jobs=-1,
             param_grid={'decisiontreeregressor__max_depth': [1, 2, 3, 4, 5, 6,
                                                              7, 8, 9, 10,
                                                              None],
                         'decisiontreeregressor__min_samples_leaf': [1, 2, 3, 4,
                                                                     5, 6, 7, 8,
                                                                     9, 10],
                         'decisiontreeregressor__min_samples_split': [2, 3, 4]},
             verbose=1)

In [ ]:
# Obtain the best combination directly
grid_search.best_params_

{'decisiontreeregressor__max_depth': 10,
 'decisiontreeregressor__min_samples_leaf': 8,
 'decisiontreeregressor__min_samples_split': 2}

**Define the Best Model**

In [ ]:
# Now define the best version of the model
best_model = grid_search.best_estimator_

**Predict and Evaluate**

In [ ]:
# Predict and Evaluate with custom function
evaluate_regression(best_model, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 15,556.335
- MSE = 627,110,888.699
- RMSE = 627,110,888.699
- R^2 = 0.909

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 21,261.349
- MSE = 1,002,114,510.787
- RMSE = 1,002,114,510.787
- R^2 = 0.795


#**الخطوة 2: بناء الهيكل التنظيمي**

In [ ]:
import os
# إنشاء مجلد للنماذج ومجلد فرعي خاص ببيانات Autompg
# سننشئ المجلدات التي ستحتوي على النماذج.
os.makedirs('/content/drive/MyDrive/AXSOSACADEMY/02-IntroML/Week06/Models', exist_ok=True)

In [ ]:
# Confirm creation of new folder
os.listdir('/content/drive/MyDrive/AXSOSACADEMY/02-IntroML/Week06/Models')

[]

In [ ]:
# Add Autompg subfolder to Models
os.makedirs('/content/drive/MyDrive/AXSOSACADEMY/02-IntroML/Week06/Models/Autompg',exist_ok=True)
# Confirm creation of new folder
os.listdir('/content/drive/MyDrive/AXSOSACADEMY/02-IntroML/Week06/Models/Autompg')

[]

In [ ]:
# هذا الكود وظيفته البحث عن أفضل "إعدادات" (Hyperparameters) لموديل الـ Decision Tree لضمان أعلى دقة ممكنة، ومن ثم حفظ النتائج.
## Example GridSearch code from previous lesson
# Parameters to try
param_grid = {'decisiontreeregressor__max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, None],
              'decisiontreeregressor__min_samples_leaf': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
              'decisiontreeregressor__min_samples_split': [2, 3, 4]}
# Instantiate and fit the gridsearch
grid_search = GridSearchCV(dec_tree_pipe, param_grid, n_jobs = -1, verbose = 1)
grid_search.fit(X_train, y_train)
## Evaluate the best model
best_model = grid_search.best_estimator_
evaluate_regression(best_model, X_train, y_train, X_test, y_test)


Fitting 5 folds for each of 330 candidates, totalling 1650 fits
------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 15,556.335
- MSE = 627,110,888.699
- RMSE = 627,110,888.699
- R^2 = 0.909

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 21,261.349
- MSE = 1,002,114,510.787
- RMSE = 1,002,114,510.787
- R^2 = 0.795


In [ ]:
# بدل ما تحفظ كل متغير لحاله، أنت بتعمل قاموس (Dictionary) بتسميه export وبتحط فيه كل شي اشتغلت عليه
## saving variables for next lesson/notebook
import joblib
## creating a dictionary of all of the variables to save for later
export = {'X_train':X_train,
         'y_train': y_train,
         'X_test':X_test,
          "y_test": y_test,
         'preprocessor':preprocessor,
         'GridSearch':grid_search}

In [ ]:
# saving the export dict as a joblib file--saving to new Models/Autompg folder
joblib.dump(export, '/content/drive/MyDrive/AXSOSACADEMY/02-IntroML/Week06/Models/Autompg/dec_tree_gridsearch.joblib')

['/content/drive/MyDrive/AXSOSACADEMY/02-IntroML/Week06/Models/Autompg/dec_tree_gridsearch.joblib']

In [ ]:
# Confirm the file was saved by loading it back in
loaded = joblib.load('/content/drive/MyDrive/AXSOSACADEMY/02-IntroML/Week06/Models/Autompg/dec_tree_gridsearch.joblib')
loaded.keys()

dict_keys(['X_train', 'y_train', 'X_test', 'y_test', 'preprocessor', 'GridSearch'])